In [1]:
from conch.open_clip_custom import create_model_from_pretrained
import torch
import os
from PIL import Image
from pathlib import Path
# remember to "conda activate conch"

/home/linus-dannull/anaconda3/envs/conch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/linus-dannull/anaconda3/envs/conch/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
model_cfg = 'conch_ViT-B-16'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
checkpoint_path = 'checkpoints/conch/pytorch_model.bin'
model, preprocess = create_model_from_pretrained(model_cfg, checkpoint_path, device=device)
_ = model.eval()



In [3]:
image = Image.open("mhist_data/images/MHIST_aaa.png")
image_tensor = preprocess(image).unsqueeze(0).to(device)
# image.resize((224, 224)) # MHIST already are 224x224

In [4]:
from conch.open_clip_custom import tokenize, get_tokenizer
tokenizer = get_tokenizer()
classes = ['sessile serrated adenomas/polyps', 
           'hyperplastic polyps']
prompts = ['an H&E image of sessile serrated adenomas/polyps', 
           'an H&E image of hyperplastic polyps']



In [8]:
# had to ensure I was using proper library versions for "transformers" and "tokenizers", executed the below pip command for this
# pip install "transformers<4.30" "tokenizers<0.14"
tokenized_prompts = tokenize(texts=prompts, tokenizer=tokenizer).to(device)
tokenized_prompts.shape

torch.Size([2, 128])

In [9]:
with torch.no_grad():
    image_features = model.encode_image(image_tensor)
    text_features = model.encode_text(tokenized_prompts)

In [13]:
similarities = image_features @ text_features.T   # shape: [1, num_classes]

# combined contains:
# (1) the raw image embedding (rich visual/morphological features)
# (2) similarity scores to each prompt (explicit semantic alignment)
# Together, this gives both low-level visual information and high-level
# class-specific signals that can be used by a downstream classifier.
combined = torch.cat([
    image_features,
    similarities
], dim=-1)


In [ ]:
# --- CLASS PROMPTS ---
class_prompts = [
    "sessile serrated adenomas/polyps",
    "hyperplastic polyps"
]

# --- MORPHOLOGY PROMPTS ---
morphology_prompts = [
    "serrated glandular architecture",
    "saw-tooth appearance of crypts",
    "dilated and branching crypts",
    "basal crypt dilation",
    "horizontal crypt growth",
    "uniform straight crypts",
    "well-formed round crypts",
    "regular epithelial lining",
    "mucin-rich goblet cells",
    "abundant goblet cells",
    "low-grade dysplasia",
    "distorted glandular structure",
    "crowded glands",
    "inflammatory cells in lamina propria",
    "clean background with minimal inflammation",
    "benign epithelial tissue",
    "normal colon mucosa",
]

# --- COMBINE ---
all_prompts = class_prompts + morphology_prompts

In [16]:
tokenized_prompts = tokenize(texts=all_prompts, tokenizer=tokenizer).to(device)
with torch.no_grad():
    text_features = model.encode_text(tokenized_prompts)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

In [27]:
from pathlib import Path
from PIL import Image
import torch

image_dir = Path("mhist_data/images")

results = []

count = 0
for img_path in image_dir.glob("*.png"):
    if count > 5:
        break
    count += 1
    print("img_path:", img_path)

    image = Image.open(img_path)
    image_tensor = preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        image_features = model.encode_image(image_tensor)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        similarities = image_features @ text_features.T  # [1, num_prompts]

    # --- Top-k prompts ---
    k = 5
    topk_vals, topk_idxs = similarities[0].topk(k)

    topk_prompts = [all_prompts[i] for i in topk_idxs.tolist()]
    topk_scores = topk_vals.tolist()

    # --- Class prediction (first 2 prompts) ---
    class_scores = similarities[0][:len(class_prompts)]
    class_probs = class_scores.softmax(dim=-1)

    pred_class = class_prompts[class_probs.argmax().item()]

    results.append({
        "image": img_path.name,
        "predicted_class": pred_class,
        "class_probs": class_probs.tolist(),
        "top_descriptions": topk_prompts,
        "top_scores": topk_scores
    })

img_path: mhist_data/images/MHIST_dnc.png
img_path: mhist_data/images/MHIST_cpm.png
img_path: mhist_data/images/MHIST_aay.png
img_path: mhist_data/images/MHIST_drg.png
img_path: mhist_data/images/MHIST_eej.png
img_path: mhist_data/images/MHIST_dbg.png


In [29]:
for r in results[:5]:
    print("Image:", r["image"])
    print("Predicted class:", r["predicted_class"])
    print("Class probs:", r["class_probs"])
    print("Top descriptions:")
    for desc, score in zip(r["top_descriptions"], r["top_scores"]):
        print(f"  - {desc} ({score:.3f})")
    print("-" * 50)

Image: MHIST_dnc.png
Predicted class: an H&E image of sessile serrated adenomas/polyps
Class probs: [0.5024951696395874, 0.4975048005580902]
Top descriptions:
  - an H&E image of sessile serrated adenomas/polyps (0.543)
  - an H&E image of hyperplastic polyps (0.534)
  - mucin-rich goblet cells (0.385)
  - low-grade dysplasia (0.351)
  - serrated glandular architecture (0.325)
--------------------------------------------------
Image: MHIST_cpm.png
Predicted class: an H&E image of sessile serrated adenomas/polyps
Class probs: [0.518146812915802, 0.4818531572818756]
Top descriptions:
  - an H&E image of sessile serrated adenomas/polyps (0.671)
  - an H&E image of hyperplastic polyps (0.599)
  - mucin-rich goblet cells (0.403)
  - abundant goblet cells (0.396)
  - clean background with minimal inflammation (0.394)
--------------------------------------------------
Image: MHIST_aay.png
Predicted class: an H&E image of sessile serrated adenomas/polyps
Class probs: [0.5032317042350769, 0.496